In [1]:
# Replace these values with your actual API credentials
client_id = 'DF26CD57LX-100'
secret_key = '5SS5D9X5TR'
redirect_uri = 'https://fessorpro.com/'
response_type = "code"  
state = "sample_state"


secondary_data=True
if secondary_data:
    api_key = 'z59mkhj6yg8b6c81'
    access_token_kite = 'jnQRoinTwJNGYPrCoa5k0nuVKCtdGzXY'

threshold=12
take_type={'FIXED':1,'PERCENTAGE':2,'RANGE':3,'ATR':4}
T1 = 'RANGE'; v1 = 0.3
T2 = 'RANGE'; v2 = 1    ;v2_pos=0.5
T3 = 'RANGE'; v3 = 1.5  ;v3_pos=0.5
T4 = 'RANGE'; v4 = 2    ;v4_pos=0
T5 = 'RANGE'; v5 = 3    ;v5_pos=0

#ATR 2 
wait_buffer=120
lot_size=65
quantity_position=2
candle_1='5'
candle_2='15'
candle_3='60'

min_15_condition=True
min_60_condition=True
index_15_condition=True

rsi_1=14;rsi_smooth_1=14
rsi_2=14;rsi_smooth_2=14
rsi_3=14;rsi_smooth_3=14
atr_length=14
strategy_name='nikita'
account_type='LIVE'
time_zone="Asia/Kolkata"
start_hour,start_min=9,30
end_hour,end_min=15,5

index_name='NIFTY50'
exchange='NSE'
type2='INDEX'
fyers_underlying_index=f"{exchange}:{index_name}-{type2}"
kite_index={'NIFTY50':256265,'NIFTYBANK':260105}
kite_underlying_index_token=kite_index.get(index_name)
symbol_list=['NIFTY2630225550CE','NIFTY2630225350PE']
fyers_initials='NSE:'
exchange_kite='NFO'   
access_token_fyers = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhdWQiOlsiZDoxIiwiZDoyIiwieDowIiwieDoxIiwieDoyIl0sImF0X2hhc2giOiJnQUFBQUFCcG4tTVVVS3ZPMjljRXNJcmRVczl3RjhaX0M2NzRVQmd1RVk1WDJ2UUhSYzNSX2FHdS15al9JZzg5OHNwOUFQOUVVaXpyWlNKV0doaXo3dC1VTXZYbEx2cUxsb1RMSkVXRVR1aEw0aFZMR3cyQjdHTT0iLCJkaXNwbGF5X25hbWUiOiIiLCJvbXMiOiJLMSIsImhzbV9rZXkiOiJmM2IwNzA2NmQ5Yjc5ODc5MWU0OTBjMjA0NmIyNjI5OWEyNGEyOWU3NDUwODNjNjZmYzI5Y2NiZCIsImlzRGRwaUVuYWJsZWQiOiJOIiwiaXNNdGZFbmFibGVkIjoiTiIsImZ5X2lkIjoiWFM0NTQ3NCIsImFwcFR5cGUiOjEwMCwiZXhwIjoxNzcyMTUyMjAwLCJpYXQiOjE3NzIwODYwMzYsImlzcyI6ImFwaS5meWVycy5pbiIsIm5iZiI6MTc3MjA4NjAzNiwic3ViIjoiYWNjZXNzX3Rva2VuIn0.bgZHr1njLEF0TQN5N8_fuUq4T55UwscmMVAzUdol_gE"



global final_data
final_data={symbol:{} for symbol in symbol_list}
final_data[index_name]={}

# CRITICAL: Patch signal module to prevent errors in non-main threads
# This must be done before importing fyers_apiv3 (which uses Twisted)
import signal
import threading

# Store original functions
_original_signal = signal.signal
_original_set_wakeup_fd = signal.set_wakeup_fd

def safe_signal(signalnum, handler):
    """Wrapper for signal.signal that only works in main thread"""
    if threading.current_thread() is threading.main_thread():
        return _original_signal(signalnum, handler)
    # In non-main thread, just return a dummy handler
    return handler

def safe_set_wakeup_fd(fd):
    """Wrapper for signal.set_wakeup_fd that only works in main thread"""
    if threading.current_thread() is threading.main_thread():
        return _original_set_wakeup_fd(fd)
    # In non-main thread, return -1 (indicating no previous fd)
    return -1

# Replace signal functions with our safe versions
signal.signal = safe_signal
signal.set_wakeup_fd = safe_set_wakeup_fd

# Import the required module from the fyers_apiv3 package
from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws
import pandas as pd
import pendulum as dt
import asyncio
import pickle
import time
import webbrowser
import os
import sys
import numpy as np
import certifi
from kiteconnect import KiteTicker, KiteConnect
from datetime import datetime
import logging
from concurrent.futures import ThreadPoolExecutor

#disable fyersApi and Fyers Request logs
import logging

In [2]:
# Initialize FyersModel instances for synchronous and asynchronous operations
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token_fyers, log_path=None)
fyers_asysc = fyersModel.FyersModel(client_id=client_id, is_async=True, token=access_token_fyers, log_path=None)

available_cash= fyers.funds()
print('available cash:', available_cash.get('fund_limit')[-1].get('equityAmount'))


available cash: 15992.54


In [11]:
def sell_partial_quantity(ticker, quantity_to_sell):
    """Sell a partial quantity of the position for the given ticker"""
    try:
        quantity_to_sell = int(quantity_to_sell)
        data = {
            "symbol": f"{fyers_initials}{ticker}",
            "qty": quantity_to_sell*lot_size,
            "type": 2,  # Market order
            "side": -1,  # Sell
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder": False,
            "orderTag": "algo_partial_exit",
            "isSliceOrder": False
        }
        response = fyers.place_order(data=data)
        print(response)
        logging.info(f'Partial sell order response: {response}')
    except Exception as e:
        logging.error(f'Error placing partial sell order: {e}')


In [3]:
ticker=symbol_list[0]
quantity_position=2
lot_size=65
def take_position(ticker, quantity_position):
        try:
                data = {
                                "symbol":fyers_initials+ticker,
                                "qty":lot_size*quantity_position,
                                "type":2,
                                "side":1,
                                "productType":"INTRADAY",
                                "limitPrice":0,
                                "stopPrice":0,
                                "validity":"DAY",
                                "disclosedQty":0,
                                "offlineOrder":False,
                                "orderTag":f"{int(33.99)}d{int(44.99)}",
                                "isSliceOrder":False
                        }

                response = fyers.place_order(data=data)
                print(response)
                logging.info(f'{ticker}: Order response: {response}')
        except Exception as e:
                logging.error(f'Error placing order for {ticker}: {e}')

{'code': 1101,
 'message': 'Successfully placed order',
 's': 'ok',
 'id': '26022600507913'}

In [13]:
sell_partial_quantity(ticker, 1)

DEBUG:FyersAPIRequest:{'Status Code': 200, 'API': '/orders/sync'}


{'code': 1101, 'message': 'Successfully placed order', 's': 'ok', 'id': '26022600510355'}


In [14]:
master = fyers.getSymbolMaster()        # Pulls entire symbol master
master

AttributeError: 'FyersModel' object has no attribute 'getSymbolMaster'

In [ ]:

# NSE – Currency Derivatives:
# https://public.fyers.in/sym_details/NSE_CD.csv
# NSE – Equity Derivatives:
# https://public.fyers.in/sym_details/NSE_FO.csv
# NSE – Commodity:
# https://public.fyers.in/sym_details/NSE_COM.csv
# NSE – Capital Market:
# https://public.fyers.in/sym_details/NSE_CM.csv

# MCX - Commodity:
# https://public.fyers.in/sym_details/MCX_COM.csv

In [35]:
import requests
from io import StringIO
import pandas as pd
import certifi


symbol_list=['NIFTY2630225550CE','NIFTY2630225350PE']
fyers_initials='MCX:'

def get_lot_size_for_symbols(symbol_list, fyers_initials):
    try:

        urls = [
            "https://public.fyers.in/sym_details/NSE_CM.csv",
            "https://public.fyers.in/sym_details/NSE_FO.csv",
            "https://public.fyers.in/sym_details/MCX_COM.csv",
        ]

        column_names = [
            "Fytoken", "Symbol Details", "Exchange Instrument Type", "Minimum Lot Size",
            "Tick Size", "ISIN", "Trading Session", "Last Update Date", "Expiry Date",
            "Symbol Ticker", "Exchange", "Segment", "Scrip Code", "Underlying Symbol",
            "Underlying Scrip Code", "Strike Price", "Option Type", "Underlying FyToken",
            "Reserved Column 1", "Reserved Column 2", "Reserved Column 3",
        ]

        frames = []
        for url in urls:
            try:
                r = requests.get(url, timeout=30, verify=certifi.where())
                r.raise_for_status()
            except requests.exceptions.SSLError:
                r = requests.get(url, timeout=30, verify=False)
                r.raise_for_status()
            frames.append(pd.read_csv(StringIO(r.text), header=None, names=column_names))

        combined_df = pd.concat(frames, ignore_index=True)
        symbol_details = combined_df["Symbol Details"].astype(str)
        symbol_ticker = combined_df["Symbol Ticker"].astype(str)

        final_lot_size = {}
        for symbol in symbol_list:
            symbol_with_prefix = f"{fyers_initials}{symbol}"

            match = combined_df[symbol_details.eq(symbol_with_prefix)]
            if match.empty:
                match = combined_df[symbol_details.eq(symbol)]
            if match.empty:
                match = combined_df[symbol_ticker.eq(symbol_with_prefix)]
            if match.empty:
                match = combined_df[symbol_ticker.str.contains(symbol, na=False)]

            if match.empty:
                raise LookupError(f"Lot size not found for symbol: {symbol}")

            size = match["Minimum Lot Size"].iloc[0]
            if pd.isna(size):
                raise ValueError(f"Invalid lot size for symbol: {symbol}")

            final_lot_size[symbol] = int(size)

        return final_lot_size

    except Exception as e:
        raise SystemExit(f"Execution stopped: {e}")

get_lot_size_for_symbols(symbol_list, fyers_initials)

{'NIFTY2630225550CE': 65, 'NIFTY2630225350PE': 65}